# ThermoFuzzy-PID · 02 · Mamdani Fuzzy PID & benchmark

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fobos123deimos/Fuzzy-Burn-in/blob/main/notebooks/02_fuzzy_pid_and_benchmark.ipynb)

Faithful transcription of the sparse rules in Figure 15, an experimental rule-completion variant and closed-loop comparisons with IMC-PID.

> **Data provenance:** experiments use deterministic synthetic thermal signals and parameters reported in a 2021 undergraduate thesis. No original sensor time series have been recovered; numerical results do not reproduce or validate the published curves.

In [ ]:
# Import local editable source, or install the repository in Google Colab.
from pathlib import Path
import sys, subprocess
candidates = [Path.cwd() / "src", Path.cwd().parent / "src"]
local_src = next((p for p in candidates if (p / "burnin_pid_fuzzy").exists()), None)
if local_src is not None:
    sys.path.insert(0, str(local_src.resolve()))
else:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "git+https://github.com/fobos123deimos/Fuzzy-Burn-in.git"])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from burnin_pid_fuzzy.models import (Plant, Scenario, DoorEvent, PLANTS_FROM_THESIS,
                                      fopdt_step_response, simulate, simulate_methods)
from burnin_pid_fuzzy.controllers import FuzzyPID, FixedPID, imc_pid_gains
from burnin_pid_fuzzy.fuzzy import FuzzyRuleBase, infer_normalized_gains, membership_degrees
from burnin_pid_fuzzy.metrics import evaluate
np.set_printoptions(precision=4, suppress=True)


## 1. Controller equations and representation

The independent PID implementation is

$$u_{\mathrm{raw}}[k]=K_p[k]e[k]+I[k]-K_d[k]\dot y_f[k],\qquad e[k]=r[k]-y[k],\tag{4}$$
$$I[k]=I[k-1]+K_i[k]\Delta t\,e[k].\tag{5}$$

The derivative is on the **measurement**, filtered to avoid setpoint kick. The control is constrained to $u\in[0,10]$ with **conditional anti-windup**. These implementation choices are openly documented extensions, not exact replications of the original Simulink block.

For a reference IMC-PID, the undergraduate thesis (Table 2) uses

$$K_p=\frac{\tau+\theta/2}{K(\lambda+\theta)},\quad T_i=\tau+\theta/2,\quad T_d=\frac{\tau\theta}{2\tau+\theta}.$$

Thus $K_i=K_p/T_i$, $K_d=K_pT_d$ in the independent form. The paper reports $K_p\approx0.4640$, $T_i\approx156.0254$, $T_d\approx4.2655$ for $\lambda=35.3273$.

In [ ]:
p=PLANTS_FROM_THESIS["mean_temperature"]
imc=imc_pid_gains(p.K,p.tau,p.delay,lam=35.3273)
print(imc)
print("ISA form:","Kp=",round(imc.kp,4),"Ti=",round(imc.kp/imc.ki,4),"Td=",round(imc.kd/imc.kp,4))

## 2. Fuzzification and M-o-M output

The thesis-based fuzzy inference system has two inputs: $|e|\in[0,40]$, $\Delta e\in[-40,40]$ and three normalized outputs $K'_p,K'_i,K'_d\in[0,1]$ (Eqs. 2.13–2.14). Each input has **7** linguistic terms, and each output has **5**.

For each rule $j$, the min-AND firing strength is $w_j=\min(\mu_{A_j}(|e|),\mu_{B_j}(\Delta e))$. Aggregation of clipped Mamdani output sets is

$$\mu_{\mathrm{out}}(z)=\max_j\min(w_j,\mu_{C_j}(z)).$$

The mean-of-maxima (M-o-M) defuzzified value is

$$z^*=\frac{1}{|\mathcal M|}\sum_{z\in\mathcal M}z,\quad\mathcal M=\arg\max_{z\in\{0,.005,\dots,1\}}\mu_{\mathrm{out}}(z).\tag{6}$$

Output gain mapping: $K_j=K_{j,\min}+(K_{j,\max}-K_{j,\min})K'_j$. The precise MF breakpoints are **assumed equally spaced**, because the thesis illustrates but does not tabulate all coordinates.

In [ ]:
fig, ax=plt.subplots(1,2,figsize=(11,3.5))
for x0,x1,a,title in [(0,40,ax[0],"Error magnitude"),(-40,40,ax[1],"Signed delta-error")]:
    grid=np.linspace(x0,x1,250)
    for j,name in enumerate(("NB","NM","NS","Z","PS","PM","PB")):
        a.plot(grid,[membership_degrees(x,x0,x1)[j] for x in grid],label=name)
    a.set(xlabel=title,ylabel="Membership",ylim=(0,1.05));a.grid(alpha=.2)
ax[1].legend(ncol=7,loc="upper center",bbox_to_anchor=(.5,-.17),fontsize=9)
plt.tight_layout();plt.show()

## 3. Sparse rule base vs. new dense rule-completion experiment

The original Figure 15 omits all NM and PM **columns** and some isolated output gains. In sparse mode an undefined gain is retained. Dense mode interpolates neighboring consequent categories, which is a **new assumption** and must be judged experimentally.

In [ ]:
for mode in ["sparse","dense"]:
    base=FuzzyRuleBase(mode)
    print(mode,"(e=26.67, delta=-40):",infer_normalized_gains(40*4/6,-40,rule_base=base,previous=np.array([.5,.17,.6])).round(4))
E=np.linspace(0,40,61)
fig,ax=plt.subplots(1,3,figsize=(12,3.3),sharex=True)
for mode,style in [("sparse","--"),("dense","-")]:
    values=np.array([infer_normalized_gains(e,0,rule_base=FuzzyRuleBase(mode)) for e in E])
    for j,name in enumerate(["Kp'","Ki'","Kd'"]):
        ax[j].plot(E,values[:,j],style,label=mode)
        ax[j].set(xlabel="|error| (°C)",ylabel=name,ylim=(0,1));ax[j].grid(alpha=.2)
ax[0].legend();plt.tight_layout();plt.show()

## 4. Closed-loop experiment: original sparse vs. proposed variants

The original thesis specifies a 65 °C setpoint, ~3750 s identification horizon and door perturbations around 2000 and 3000 s. Here we use explicitly **synthetic** 30 °C ambient, effective heater output 0–10, and two assumed -5 input-unit door events lasting 60 and 300 s. This is *not* the original test bench.

In [ ]:
scenario=Scenario()
runs=simulate_methods(scenario=scenario)
summary=pd.DataFrame([{"controller":name,**evaluate(result,scenario)} for name,result in runs.items()])
print(summary.round(3).to_string(index=False))
fig,axes=plt.subplots(2,1,figsize=(11,7),sharex=True)
for name,result in runs.items():
    axes[0].plot(result["time_s"],result["measurement_c"],label=name)
    axes[1].plot(result["time_s"],result["control"],label=name)
axes[0].axhline(65,ls="--",color="black",lw=1)
for a in axes:
    for d in scenario.doors:a.axvspan(d.start,d.start+d.duration,color="gray",alpha=.12)
    a.grid(alpha=.2);a.legend()
axes[0].set_ylabel("Temperature (°C)");axes[1].set_ylabel("Heater units")
axes[1].set_xlabel("Time (s)");plt.tight_layout();plt.show()

## 5. Reading these results correctly

The 2021 undergraduate thesis *reported* a roughly **22%** faster time-to-60 °C and a **3%** faster disturbance recovery for its PID-Fuzzy versus IMC, based on its own Simulink cases (Section 4). The fresh synthetic benchmark above can give **different rankings**, due to assumed actuator saturation, disturbance strengths, sensor model, fuzzy memberships, anti-windup, derivative treatment and missing original measurements. Do not present these outputs as independent confirmation of the published percentages.

**Next:** [03 · Robustness](03_uncertainty_and_robustness.ipynb).